# Intro - coppied notebook and changed --- check all text sections to ensure accuracy 

This is a deep learning based Geo MLP model to predict city level crime risk from a mix of geographic coordinates and country level features. The goal is to estimate a crime index for any given latitude/longitude and country, even in places without direct city level data, by leveraging patterns in national crime indices, demographics, and population density. This first baseline model deliberately excludes city level safety scores to avoid trivial shortcuts, forcing the network to learn how far country context and location alone can go in explaining local risk.

In [1]:
# City Level Safety Index Predictor with Geo-MLP Model 
import os
import json
import copy
import random
import pickle

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

RANDOM_STATE = 42

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything(RANDOM_STATE)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

artifact_dir = "artifacts/geo_mlp_safety_lower_bound_v1"
os.makedirs(artifact_dir, exist_ok=True)

Device: cpu


In [2]:
# Load data -----> lon/lat,safteyandcrimeindex are city level all other are country level   
 
city_path = "data/compiled_model_ready/MR_cities_with_country_v1.csv"
cities = pd.read_csv(city_path)

print("Columns:", cities.columns.tolist())
print(cities.head())
print("Number of cities:", len(cities))

Columns: ['city', 'country', 'country_norm', 'lat', 'lon', 'crime_index', 'safety_index', 'crimeindex_2023', 'crimeindex_2020', 'safetyindex_2020', 'age_0_14', 'age_15_64', 'age_65_plus', 'population', 'density_per_km2']
           city           country      country_norm        lat         lon  \
0       Caracas         Venezuela         venezuela  10.506093  -66.914601   
1      Pretoria      South Africa      south africa -25.745928   28.187910   
2        Durban      South Africa      south africa -29.861825   31.009909   
3  Port Moresby  Papua New Guinea  papua new guinea  -9.474330  147.159950   
4  Johannesburg      South Africa      south africa -26.205000   28.049722   

   crime_index  safety_index  crimeindex_2023  crimeindex_2020  \
0         83.6          16.4            83.76            84.49   
1         82.0          18.0            76.86            77.49   
2         81.0          19.0            76.86            77.49   
3         80.7          19.3            80.79 

In [3]:
cities.shape

(509, 15)

In [4]:
#  Feature selection and target

feature_cols = [
    "lat",
    "lon",
    # core country risk signals
    "crimeindex_2020",
    "crimeindex_2023",
    "safetyindex_2020",
    # demographics & scale
    "age_0_14",
    "age_15_64",
    "age_65_plus",
    "population",
    "density_per_km2",
    
]

target_col = "safety_index"  

# Drop any rows missing required features or target 
cities_model = cities.dropna(subset=feature_cols + [target_col]).copy()

X = cities_model[feature_cols].values.astype(np.float32)
y = cities_model[target_col].values.astype(np.float32)

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Target:", target_col)

X shape: (509, 10)
y shape: (509,)
Target: safety_index


In [5]:
# Train/test split and scaling

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Train:", X_train_scaled.shape, "Test:", X_test_scaled.shape)

Train: (407, 10) Test: (102, 10)


In [6]:
# Tensor datasets and DataLoaders

X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32).view(-1, 1)

X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32).view(-1, 1)

print("Train:", X_train_tensor.shape, y_train_tensor.shape)
print("Test:", X_test_tensor.shape, y_test_tensor.shape)

Train: torch.Size([407, 10]) torch.Size([407, 1])
Test: torch.Size([102, 10]) torch.Size([102, 1])


In [7]:
# DataLoader helper

def make_loaders(X_train_tensor, y_train_tensor, X_test_tensor, y_test_tensor, batch_size=32):
    train_ds = TensorDataset(X_train_tensor, y_train_tensor)
    test_ds = TensorDataset(X_test_tensor, y_test_tensor)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_ds, batch_size=64, shuffle=False)

    return train_loader, test_loader

In [8]:
# Geo MLP model definition - flexible

class GeoMLP(nn.Module):
    def __init__(self, input_dim, hidden_dims=(64, 32), dropout=0.2):
        super().__init__()

        layers = []
        prev_dim = input_dim

        for hidden_dim in hidden_dims:
            layers.append(nn.Linear(prev_dim, hidden_dim))
            layers.append(nn.ReLU())
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            prev_dim = hidden_dim

        layers.append(nn.Linear(prev_dim, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

In [9]:
# Evaluation helper

def evaluate(model, loader):
    model.eval()
    preds = []
    targets = []

    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device)
            yb = yb.to(device)

            out = model(xb)
            preds.append(out.cpu().numpy())
            targets.append(yb.cpu().numpy())

    preds = np.vstack(preds).ravel()
    targets = np.vstack(targets).ravel()

    rmse = np.sqrt(mean_squared_error(targets, preds))
    mae = mean_absolute_error(targets, preds)
    r2 = r2_score(targets, preds)

    return rmse, mae, r2, preds, targets

In [10]:
# Train one variant

def train_one_model(
    model_name,
    hidden_dims=(64, 32),
    dropout=0.2,
    lr=1e-3,
    weight_decay=1e-4,
    batch_size=32,
    n_epochs=300,
    verbose=True
):
    seed_everything(RANDOM_STATE)

    train_loader, test_loader = make_loaders(
        X_train_tensor, y_train_tensor, X_test_tensor, y_test_tensor, batch_size=batch_size
    )

    model = GeoMLP(
        input_dim=X_train_tensor.shape[1],
        hidden_dims=hidden_dims,
        dropout=dropout
    ).to(device)

    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    best_rmse = float("inf")
    best_state = None
    history = []

    for epoch in range(1, n_epochs + 1):
        model.train()

        for xb, yb in train_loader:
            xb = xb.to(device)
            yb = yb.to(device)

            optimizer.zero_grad()
            preds = model(xb)
            loss = criterion(preds, yb)
            loss.backward()
            optimizer.step()

        if epoch % 20 == 0 or epoch == n_epochs:
            train_rmse, train_mae, train_r2, _, _ = evaluate(model, train_loader)
            test_rmse, test_mae, test_r2, _, _ = evaluate(model, test_loader)

            history.append({
                "epoch": epoch,
                "train_rmse": train_rmse,
                "train_mae": train_mae,
                "train_r2": train_r2,
                "test_rmse": test_rmse,
                "test_mae": test_mae,
                "test_r2": test_r2,
            })

            if verbose:
                print(
                    f"{model_name} | Epoch {epoch:03d} | "
                    f"Train RMSE {train_rmse:.2f}, R2 {train_r2:.3f} | "
                    f"Test RMSE {test_rmse:.2f}, R2 {test_r2:.3f}"
                )

            if test_rmse < best_rmse:
                best_rmse = test_rmse
                best_state = copy.deepcopy(model.state_dict())

    if best_state is not None:
        model.load_state_dict(best_state)

    final_rmse, final_mae, final_r2, preds, targets = evaluate(model, test_loader)

    result = {
        "model_name": model_name,
        "hidden_dims": hidden_dims,
        "dropout": dropout,
        "lr": lr,
        "weight_decay": weight_decay,
        "batch_size": batch_size,
        "n_epochs": n_epochs,
        "best_test_rmse": final_rmse,
        "best_test_mae": final_mae,
        "best_test_r2": final_r2,
        "best_state_dict": copy.deepcopy(model.state_dict()),
        "history": history,
        "preds": preds,
        "targets": targets,
    }

    return result

In [11]:
# Variant grid

variant_configs = [
    {
        "model_name": "safety_lb_v1_e200",
        "hidden_dims": (64, 32),
        "dropout": 0.2,
        "lr": 1e-3,
        "weight_decay": 1e-4,
        "batch_size": 32,
        "n_epochs": 200,
    },
    {
        "model_name": "safety_lb_v1_e300",
        "hidden_dims": (64, 32),
        "dropout": 0.2,
        "lr": 1e-3,
        "weight_decay": 1e-4,
        "batch_size": 32,
        "n_epochs": 300,
    },
    {
        "model_name": "safety_lb_v1_e400",
        "hidden_dims": (64, 32),
        "dropout": 0.2,
        "lr": 1e-3,
        "weight_decay": 1e-4,
        "batch_size": 32,
        "n_epochs": 400,
    },
    {
        "model_name": "safety_lb_v1_low_dropout",
        "hidden_dims": (64, 32),
        "dropout": 0.1,
        "lr": 1e-3,
        "weight_decay": 1e-4,
        "batch_size": 32,
        "n_epochs": 300,
    },
    {
        "model_name": "safety_lb_v1_higher_wd",
        "hidden_dims": (64, 32),
        "dropout": 0.2,
        "lr": 1e-3,
        "weight_decay": 5e-4,
        "batch_size": 32,
        "n_epochs": 300,
    },
    {
        "model_name": "safety_lb_v1_small",
        "hidden_dims": (32, 16),
        "dropout": 0.2,
        "lr": 1e-3,
        "weight_decay": 1e-4,
        "batch_size": 32,
        "n_epochs": 300,
    },
    {
        "model_name": "safety_lb_v1_wide",
        "hidden_dims": (128, 64),
        "dropout": 0.2,
        "lr": 1e-3,
        "weight_decay": 1e-4,
        "batch_size": 32,
        "n_epochs": 300,
    },
    {
        "model_name": "safety_lb_v1_deeper",
        "hidden_dims": (128, 64, 32),
        "dropout": 0.2,
        "lr": 1e-3,
        "weight_decay": 1e-4,
        "batch_size": 32,
        "n_epochs": 300,
    },
    {
        "model_name": "safety_lb_v1_narrow_deep",
        "hidden_dims": (64, 32, 16),
        "dropout": 0.2,
        "lr": 1e-3,
        "weight_decay": 1e-4,
        "batch_size": 32,
        "n_epochs": 300,
    },
    {
        "model_name": "safety_lb_v1_wide_low_dropout",
        "hidden_dims": (128, 64),
        "dropout": 0.1,
        "lr": 1e-3,
        "weight_decay": 1e-4,
        "batch_size": 32,
        "n_epochs": 300,
    },
]

In [12]:
# Run all variants

all_results = []

for cfg in variant_configs:
    print("\n" + "=" * 80)
    print("Training:", cfg["model_name"])
    print("=" * 80)

    result = train_one_model(**cfg, verbose=True)
    all_results.append(result)


Training: safety_lb_v1_e200
safety_lb_v1_e200 | Epoch 020 | Train RMSE 15.21, R2 -0.000 | Test RMSE 15.35, R2 0.032
safety_lb_v1_e200 | Epoch 040 | Train RMSE 11.93, R2 0.385 | Test RMSE 12.11, R2 0.398
safety_lb_v1_e200 | Epoch 060 | Train RMSE 11.05, R2 0.472 | Test RMSE 11.54, R2 0.453
safety_lb_v1_e200 | Epoch 080 | Train RMSE 10.58, R2 0.516 | Test RMSE 11.17, R2 0.488
safety_lb_v1_e200 | Epoch 100 | Train RMSE 10.22, R2 0.548 | Test RMSE 10.93, R2 0.509
safety_lb_v1_e200 | Epoch 120 | Train RMSE 10.12, R2 0.557 | Test RMSE 10.79, R2 0.522
safety_lb_v1_e200 | Epoch 140 | Train RMSE 10.05, R2 0.563 | Test RMSE 10.71, R2 0.529
safety_lb_v1_e200 | Epoch 160 | Train RMSE 9.92, R2 0.575 | Test RMSE 10.80, R2 0.521
safety_lb_v1_e200 | Epoch 180 | Train RMSE 10.03, R2 0.565 | Test RMSE 10.89, R2 0.513
safety_lb_v1_e200 | Epoch 200 | Train RMSE 9.83, R2 0.583 | Test RMSE 10.73, R2 0.527

Training: safety_lb_v1_e300
safety_lb_v1_e300 | Epoch 020 | Train RMSE 15.21, R2 -0.000 | Test RMSE 1

In [13]:
# Compare results

results_df = pd.DataFrame([
    {
        "model_name": r["model_name"],
        "hidden_dims": str(r["hidden_dims"]),
        "dropout": r["dropout"],
        "lr": r["lr"],
        "weight_decay": r["weight_decay"],
        "batch_size": r["batch_size"],
        "n_epochs": r["n_epochs"],
        "rmse": r["best_test_rmse"],
        "mae": r["best_test_mae"],
        "r2": r["best_test_r2"],
    }
    for r in all_results
])

results_df = results_df.sort_values(by="rmse", ascending=True).reset_index(drop=True)

print(results_df)

                      model_name    hidden_dims  dropout     lr  weight_decay  \
0            safety_lb_v1_deeper  (128, 64, 32)      0.2  0.001        0.0001   
1              safety_lb_v1_e400       (64, 32)      0.2  0.001        0.0001   
2             safety_lb_v1_small       (32, 16)      0.2  0.001        0.0001   
3       safety_lb_v1_low_dropout       (64, 32)      0.1  0.001        0.0001   
4              safety_lb_v1_wide      (128, 64)      0.2  0.001        0.0001   
5              safety_lb_v1_e300       (64, 32)      0.2  0.001        0.0001   
6         safety_lb_v1_higher_wd       (64, 32)      0.2  0.001        0.0005   
7              safety_lb_v1_e200       (64, 32)      0.2  0.001        0.0001   
8  safety_lb_v1_wide_low_dropout      (128, 64)      0.1  0.001        0.0001   
9       safety_lb_v1_narrow_deep   (64, 32, 16)      0.2  0.001        0.0001   

   batch_size  n_epochs       rmse       mae        r2  
0          32       300  10.543942  8.303670  0.543

In [14]:
# Save best model + artifacts

best_result = min(all_results, key=lambda x: x["best_test_rmse"])

print("\nBest model:")
print("Name:", best_result["model_name"])
print("RMSE:", round(best_result["best_test_rmse"], 4))
print("MAE:", round(best_result["best_test_mae"], 4))
print("R2:", round(best_result["best_test_r2"], 4))

artifact_dir = "artifacts/geo_mlp_safety_v1"

# save weights
weights_path = os.path.join(artifact_dir, f"{best_result['model_name']}_best_weights.pt")
torch.save(best_result["best_state_dict"], weights_path)

# save scaler
scaler_path = os.path.join(artifact_dir, f"{best_result['model_name']}_scaler.pkl")
with open(scaler_path, "wb") as f:
    pickle.dump(scaler, f)

# save config
config_to_save = {
    "model_name": best_result["model_name"],
    "feature_cols": feature_cols,
    "target_col": target_col,
    "hidden_dims": list(best_result["hidden_dims"]),
    "dropout": best_result["dropout"],
    "lr": best_result["lr"],
    "weight_decay": best_result["weight_decay"],
    "batch_size": best_result["batch_size"],
    "n_epochs": best_result["n_epochs"],
    "rmse": best_result["best_test_rmse"],
    "mae": best_result["best_test_mae"],
    "r2": best_result["best_test_r2"],
    "data_path": city_path,
    "random_state": RANDOM_STATE,
     "notes": "Lower-bound safety baseline with no city-level crime feature."
}

config_path = os.path.join(artifact_dir, f"{best_result['model_name']}_config.json")
with open(config_path, "w") as f:
    json.dump(config_to_save, f, indent=2)

# save results table
results_path = os.path.join(artifact_dir, "geo_mlp_safety_lower_bound_variant_results.csv")
results_df.to_csv(results_path, index=False)

print("\nSaved:")
print(weights_path)
print(scaler_path)
print(config_path)
print(results_path)


Best model:
Name: safety_lb_v1_deeper
RMSE: 10.5439
MAE: 8.3037
R2: 0.5434

Saved:
artifacts/geo_mlp_safety_v1\safety_lb_v1_deeper_best_weights.pt
artifacts/geo_mlp_safety_v1\safety_lb_v1_deeper_scaler.pkl
artifacts/geo_mlp_safety_v1\safety_lb_v1_deeper_config.json
artifacts/geo_mlp_safety_v1\geo_mlp_safety_lower_bound_variant_results.csv


In [15]:
# Reload best saved model later

with open(config_path, "r") as f:
    saved_config = json.load(f)

with open(scaler_path, "rb") as f:
    loaded_scaler = pickle.load(f)

best_model = GeoMLP(
    input_dim=len(saved_config["feature_cols"]),
    hidden_dims=tuple(saved_config["hidden_dims"]),
    dropout=saved_config["dropout"]
).to(device)

best_model.load_state_dict(torch.load(weights_path, map_location=device))
best_model.eval()

print("Reloaded:", saved_config["model_name"])

Reloaded: safety_lb_v1_deeper


In [16]:
# Predict safety for a city already in dataset

def predict_safety_for_city(city_name, country_name=None):
    city_name_norm = str(city_name).strip().lower()

    df = cities_model.copy()
    df["city_norm"] = df["city"].str.strip().str.lower()

    row_matches = df[df["city_norm"] == city_name_norm]

    if country_name is not None:
        country_name_norm = str(country_name).strip().lower()
        row_matches = row_matches[
            row_matches["country"].str.strip().str.lower() == country_name_norm
        ]

    if row_matches.empty:
        raise ValueError(f"City '{city_name}' not found in dataset")

    row = row_matches.iloc[0]

    x_raw = row[feature_cols].values.astype(np.float32).reshape(1, -1)
    x_scaled = loaded_scaler.transform(x_raw)
    x_tensor = torch.tensor(x_scaled, dtype=torch.float32).to(device)

    with torch.no_grad():
        pred = best_model(x_tensor).cpu().numpy().ravel()[0]

    return {
        "city": row["city"],
        "country": row["country"],
        "true_safety_index": float(row[target_col]),
        "predicted_safety_index": float(pred)
    }

In [17]:
result = predict_safety_for_city("Cape Town", "South Africa")
print(result)

{'city': 'Cape Town', 'country': 'South Africa', 'true_safety_index': 26.4, 'predicted_safety_index': 22.235078811645508}


In [18]:
# Inference helper for safety

country_path = "data/compiled_model_ready/country_features_2022_2023.csv"
country_df = pd.read_csv(country_path)
country_df["country_norm"] = country_df["country"].str.strip().str.lower()

def build_feature_vector_for_safety(lat, lon, country_name):
    cname = str(country_name).strip().lower()
    row = country_df[country_df["country_norm"] == cname]

    if row.empty:
        raise ValueError(f"Country '{country_name}' not found in country feature table")

    r = row.iloc[0]

    feat_vals = []
    for col in feature_cols:
        if col == "lat":
            feat_vals.append(float(lat))
        elif col == "lon":
            feat_vals.append(float(lon))
        else:
            feat_vals.append(float(r[col]))

    feat = np.array([feat_vals], dtype=np.float32)
    feat_scaled = loaded_scaler.transform(feat)
    return torch.tensor(feat_scaled, dtype=torch.float32)

def predict_safety_global(lat, lon, country_name):
    x = build_feature_vector_for_safety(lat, lon, country_name).to(device)

    best_model.eval()
    with torch.no_grad():
        pred = best_model(x).cpu().numpy().ravel()[0]

    return float(pred)

In [19]:
example_pred = predict_safety_global(-33.92, 18.42, "south africa")
print("Predicted safety_index near Cape Town:", example_pred)

Predicted safety_index near Cape Town: 22.233787536621094


# shouldnt have to run again, if copied to another model version add features and change name to be model version specific  

# End Model --------------------------------------------------------Start Report

# Summary 

The initial results are genuinely encouraging with no city level safety feature, the model still achieves a test RMSE around 11 and captures roughly half of the variance in city crime, which is impressive given it’s only using country level context plus coordinates. Even more notable, this performance comes from the very first training run no architecture tuning, no hyperparameter search, and no feature engineering beyond the cleaned dataset. I’m going to snapshot this version as a baseline and then work on a tuned copy, where I can experiment with additional city level features, regularization, and architecture tweaks to push accuracy 

# Next Steps

The next phase is all about strengthening the city‑level signal and then tuning the Geo‑MLP on top of that richer feature space.

### Add more city‑level features

Add city‑specific population (even if partially manual), and from that derive simple city density (population / built‑up area or a rough proxy).

Incorporate city‑level pollution indicators where available (e.g., PM2.5, air quality index); these often correlate with urbanization and can proxy for environmental and economic conditions.

Consider simple urban form proxies like “is_capital”, “large_city_flag”, or size buckets (small/medium/large) derived from population.

### Introduce spatial/region encoding beyond raw lat/lon

Create a country code feature that is purely numeric: e.g., an integer ID, or a 2–3 digit code.

You can go further and define a grid code over each country: partition the country into a small grid and assign each city a cell ID; this lets the model learn within‑country regional patterns (north vs south, coastal vs inland) even before you add more complex spatial features.

Optionally one‑hot or embed these codes (especially the grid or regional codes) to help the model separate distinct zones.

### Expand environmental and contextual features

Add pollution data at the city level where possible, and consider augmenting with proxies like elevation, distance to coast, or distance to the capital (derived from lat/lon).

If you later pull in open data (e.g., POIs, nightlights, or land‑use), these can become powerful city‑level predictors layered onto the current framework.

### Hyperparameter and architecture tuning on a copied notebook

Clone this baseline notebook and experiment with:

Hidden layer sizes (e.g., 64→32 vs 128→64→32),

Dropout rates (0.1–0.4),

Weight decay and learning rates.

Use the same train/test split and log RMSE/R² for each run so you can compare against the current baseline and against KNN.

Benchmark against non DL baselines

Fit a strong tree based regressor (e.g., gradient boosting) on the same features/split and compare RMSE/R².

Use that as a reference to judge whether further DL tuning is giving real gains, or if most of the value is coming from feature quality rather than model class.

# DataSet Information

To build the dataset for the Geo‑MLP, I started by integrating five separate data sources into a single, city‑level table. This involved loading each dataset, standardizing column names, and joining them on shared keys such as country and city names. I carefully normalized country strings (lowercasing, trimming whitespace, handling variants like “United States” vs “USA”) to ensure a clean join, and I resolved duplicate or conflicting records so that each city ended up with one consistent row.

Next, I enriched the city table with geographic information by adding precise latitude and longitude coordinates for every location. This step makes the dataset “geo‑aware” and enables the model to learn spatial patterns in crime risk rather than relying solely on country averages. Alongside the coordinates, I merged in a comprehensive set of country‑level features for each city: crime and safety indices (for 2020 and 2023), demographic age splits (0–14, 15–64, 65+), total population, and population density per square kilometer. Where some countries were missing specific metrics (for example, 2020 crime or safety values for a few cases), I filled them using principled rules: copying 2023 crime into 2020 when that made sense, and, in rare edge cases, hard‑coding well‑researched safety values.

I then systematically identified and resolved missing values. Rather than blindly median‑filling, I inspected the small set of remaining NaNs city by city (e.g., San Juan, Montevideo, Tashkent, Kigali) and went back to external sources to look up accurate population, demographic, and density values. This meant manually inserting exact figures for those countries so the final dataset would be numerically complete without synthetic imputation. I also verified that each city row had a full, consistent set of country features and that no column still contained missing data before saving.

Throughout this process, I kept the data in a single, modeling‑ready dataframe: one row per city, with all city‑level fields (name, country, lat, lon, crime_index, safety_index if available) and all merged country‑level attributes. After final validation—checking dtypes, ranges, and confirming zero NaNs—I saved the cleaned table as cities_model_2022_2023_with_country.csv. This file is now a high‑quality foundation for training the Geo‑MLP, giving the model a rich combination of geographic coordinates, national risk context, and demographic structure for each city in the dataset.

In [20]:
#import ipynbname

#nb_path = ipynbname.path()
#nb_name = nb_path.name 
#nb_name

#import sys
#from pathlib import Path

# PDF 
#!"{sys.executable}" -m nbconvert --to pdf "{nb_name}"